In [ ]:
# VARIANT 2: CB + LGBM + NN triple blend
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, random
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

TARGET, ID_COL, FOLDS = 'Subscribed', 'id', 5
CB_SEEDS = [42, 2024, 3407]
LGBM_SEEDS = [42, 2024]
NN_SEEDS = [42, 2024, 3407, 777, 1337]
MONTH_MAP = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}

CB_PARAMS = {'iterations': 3500, 'learning_rate': 0.01377, 'depth': 6, 'l2_leaf_reg': 3.976,
    'random_strength': 1.084, 'bagging_temperature': 0.733, 'bootstrap_type': 'Bayesian',
    'loss_function': 'Logloss', 'eval_metric': 'AUC', 'allow_writing_files': False, 'verbose': False}

LGBM_PARAMS = {'n_estimators': 2000, 'learning_rate': 0.02, 'num_leaves': 48, 'max_depth': 7,
    'min_child_samples': 25, 'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_alpha': 0.5,
    'reg_lambda': 3.0, 'class_weight': 'balanced', 'verbosity': -1, 'n_jobs': -1}

NN_PARAMS = {'emb_dim': 24, 'hidden_dim': 192, 'n_layers': 4, 'heads': 6, 'dropout': 0.2,
    'epochs': 80, 'batch_size': 1024, 'lr': 0.001, 'weight_decay': 0.01, 'patience': 14}

BASE_CAT = ['job','marital','education','default','housing','loan','contact','month','poutcome']
def rank_norm(v): return pd.Series(v).rank(method='average', pct=True).to_numpy()
def _s(d,c,l=False): v=d[c].fillna('missing').astype(str); return v.str.lower() if l else v

print('Variant 2: Triple blend (CB 60% + LGBM 15% + NN 25%)')

In [ ]:
def build_features(data):
    df = data.copy()
    if ID_COL in df.columns: df = df.drop(columns=[ID_COL])
    m,j,ma,e,ct,po,l,d,h = [_s(df,c,c=='month') for c in ['month','job','marital','education','contact','poutcome','loan','default','housing']]
    df['month'] = m
    df['month_num'] = m.map(MONTH_MAP).fillna(0).astype(np.int16)
    df['pdays_was_missing'] = (df['pdays']==-1).astype(np.int8)
    df['pdays_clean'] = df['pdays'].replace(-1, 999)
    for col in ['duration','balance','campaign','previous']: df[f'{col}_log1p'] = np.log1p(df[col].clip(lower=0))
    df['balance_abs_log1p'] = np.log1p(df['balance'].abs())
    df['pdays_log1p'] = np.log1p(df['pdays_clean'])
    df['contacts_total'] = df['campaign'] + df['previous']
    df['duration_per_campaign'] = df['duration'] / (df['campaign']+1)
    df['balance_per_age'] = df['balance'] / (df['age']+1)
    df['campaign_x_previous'] = df['campaign'] * df['previous']
    df['has_any_loan'] = ((h=='yes')|(l=='yes')).astype(np.int8)
    df['is_default'] = (d=='yes').astype(np.int8)
    df['was_contacted_before'] = (df['pdays']!=-1).astype(np.int8)
    df['is_cellular'] = (ct=='cellular').astype(np.int8)
    df['balance_signed_log1p'] = np.sign(df['balance']) * np.log1p(df['balance'].abs())
    df['balance_negative'] = (df['balance']<0).astype(np.int8)
    ps = df['pdays'].replace(-1,999)
    df['age_bucket'] = pd.cut(df['age'],[0,25,35,45,55,65,120],labels=['<=25','26-35','36-45','46-55','56-65','65+']).astype(str)
    df['campaign_bucket'] = pd.cut(df['campaign'],[-1,1,2,4,9,np.inf],labels=['1','2','3-4','5-9','10+']).astype(str)
    df['previous_bucket'] = pd.cut(df['previous'],[-1,0,1,3,np.inf],labels=['0','1','2-3','4+']).astype(str)
    df['pdays_bucket'] = pd.cut(ps,[-1,7,30,90,365,np.inf],labels=['<=1w','8-30d','31-90d','91-365d','365d+']).astype(str)
    df.loc[df['pdays']==-1,'pdays_bucket'] = 'never'
    df['duration_bucket'] = pd.cut(df['duration'],[-1,60,120,240,480,np.inf],labels=['<=1m','1-2m','2-4m','4-8m','8m+']).astype(str)
    df['day_bucket'] = pd.cut(df['day'],[0,10,20,31],labels=['early','mid','late'],include_lowest=True).astype(str)
    df['job_education'] = j+'__'+e
    df['contact_month'] = ct+'__'+m
    df['poutcome_month'] = po+'__'+m
    df['contact_day_bucket'] = ct+'__'+df['day_bucket']
    df['history_state'] = np.where(df['previous']>0, po+'__seen', 'no_previous')
    cats = BASE_CAT + ['age_bucket','campaign_bucket','previous_bucket','pdays_bucket','duration_bucket','day_bucket',
        'job_education','contact_month','poutcome_month','contact_day_bucket','history_state']
    for c in cats: df[c] = df[c].fillna('missing').astype(str)
    return df, cats

class TBlock(nn.Module):
    def __init__(s,d,h,dr): super().__init__(); s.n1=nn.LayerNorm(d); s.a=nn.MultiheadAttention(d,h,dr,batch_first=True); s.n2=nn.LayerNorm(d); s.ff=nn.Sequential(nn.Linear(d,d*2),nn.GELU(),nn.Dropout(dr),nn.Linear(d*2,d),nn.Dropout(dr))
    def forward(s,x): o,_=s.a(s.n1(x),s.n1(x),s.n1(x),need_weights=False); x=x+o; return x+s.ff(s.n2(x))
class AttnNet(nn.Module):
    def __init__(s,nn_,cd,ed,hd,nl,he,dr): super().__init__(); s.emb=nn.ModuleList([nn.Embedding(d,ed) for d in cd]); s.np=nn.Linear(nn_,hd) if nn_>0 else None; s.cp=nn.Linear(ed,hd) if cd else None; s.cls=nn.Parameter(torch.randn(1,1,hd)*0.02); s.blk=nn.ModuleList([TBlock(hd,he,dr) for _ in range(nl)]); s.head=nn.Sequential(nn.LayerNorm(hd),nn.Linear(hd,hd//2),nn.GELU(),nn.Dropout(dr),nn.Linear(hd//2,1))
    def forward(s,xn,xc): t=[]; t.append(s.cls.expand(xn.shape[0],-1,-1)); (t.append(s.np(xn).unsqueeze(1)) if s.np and xn.shape[1]>0 else None); [t.append(s.cp(s.emb[i](xc[:,i])).unsqueeze(1)) for i in range(len(s.emb))] if s.emb else None; x=torch.cat(t,1); [x:=b(x) for b in s.blk]; return torch.sigmoid(s.head(x[:,0]))

print('Models defined')

In [ ]:
def train_cb(x_tr,x_te,y,cats,params,seeds):
    oof,test=np.zeros(len(x_tr)),np.zeros(len(x_te))
    for seed in seeds:
        s_oof,s_test=np.zeros(len(x_tr)),np.zeros(len(x_te))
        for _,(ti,vi) in enumerate(StratifiedKFold(FOLDS,True,seed).split(x_tr,y)):
            m=CatBoostClassifier(**params,auto_class_weights='Balanced',random_seed=seed)
            m.fit(x_tr.iloc[ti].reset_index(drop=True),y.iloc[ti].reset_index(drop=True),eval_set=(x_tr.iloc[vi].reset_index(drop=True),y.iloc[vi].reset_index(drop=True)),cat_features=cats,use_best_model=True,early_stopping_rounds=200,verbose=False)
            s_oof[vi]=m.predict_proba(x_tr.iloc[vi].reset_index(drop=True))[:,1]; s_test+=m.predict_proba(x_te)[:,1]/FOLDS
        print(f'CB {seed}: {roc_auc_score(y,s_oof):.6f}'); oof+=s_oof/len(seeds); test+=s_test/len(seeds)
    print(f'CB Final: {roc_auc_score(y,oof):.6f}'); return oof,test

def train_lgbm(x_tr,x_te,y,cats,params,seeds):
    oof,test=np.zeros(len(x_tr)),np.zeros(len(x_te))
    xtr,xte=x_tr.copy(),x_te.copy()
    for c in cats: xtr[c]=xtr[c].astype('category'); xte[c]=xte[c].astype('category')
    for seed in seeds:
        s_oof,s_test=np.zeros(len(x_tr)),np.zeros(len(x_te))
        for _,(ti,vi) in enumerate(StratifiedKFold(FOLDS,True,seed).split(xtr,y)):
            m=LGBMClassifier(**params,random_state=seed); m.fit(xtr.iloc[ti],y.iloc[ti],eval_set=[(xtr.iloc[vi],y.iloc[vi])])
            s_oof[vi]=m.predict_proba(xtr.iloc[vi])[:,1]; s_test+=m.predict_proba(xte)[:,1]/FOLDS
        print(f'LGBM {seed}: {roc_auc_score(y,s_oof):.6f}'); oof+=s_oof/len(seeds); test+=s_test/len(seeds)
    print(f'LGBM Final: {roc_auc_score(y,oof):.6f}'); return oof,test

def train_nn(x_tr,x_te,y,cats,params,seeds):
    dev=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print(f'NN on {dev}')
    nc=[c for c in x_tr.columns if c not in cats]; cd,xct,xcte=[],np.zeros((len(x_tr),len(cats)),dtype=np.int64),np.zeros((len(x_te),len(cats)),dtype=np.int64)
    for i,c in enumerate(cats): cb=pd.concat([x_tr[c].fillna('m').astype(str),x_te[c].fillna('m').astype(str)]); co,_=pd.factorize(cb,sort=True); xct[:,i],xcte[:,i]=co[:len(x_tr)],co[len(x_tr):]; cd.append(int(co.max()+1))
    sc=StandardScaler(); xnt=sc.fit_transform(x_tr[nc].fillna(0).astype(np.float32)) if nc else np.zeros((len(x_tr),0),dtype=np.float32); xnte=sc.transform(x_te[nc].fillna(0).astype(np.float32)) if nc else np.zeros((len(x_te),0),dtype=np.float32)
    oof,test,ynp=np.zeros(len(x_tr)),np.zeros(len(x_te)),y.values.astype(np.float32)
    for seed in seeds:
        random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
        s_oof,s_test=np.zeros(len(x_tr)),np.zeros(len(x_te))
        for _,(ti,vi) in enumerate(StratifiedKFold(FOLDS,True,seed).split(x_tr,y)):
            trn,trc,try_=torch.FloatTensor(xnt[ti]).to(dev),torch.LongTensor(xct[ti]).to(dev),torch.FloatTensor(ynp[ti]).unsqueeze(1).to(dev)
            van,vac=torch.FloatTensor(xnt[vi]).to(dev),torch.LongTensor(xct[vi]).to(dev)
            ten,tec=torch.FloatTensor(xnte).to(dev),torch.LongTensor(xcte).to(dev)
            mdl=AttnNet(xnt.shape[1],cd,params['emb_dim'],params['hidden_dim'],params['n_layers'],params['heads'],params['dropout']).to(dev)
            opt=torch.optim.AdamW(mdl.parameters(),lr=params['lr'],weight_decay=params['weight_decay']); crit=nn.BCELoss(); sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,'max',patience=8,factor=0.5)
            ldr=DataLoader(TensorDataset(trn,trc,try_),batch_size=params['batch_size'],shuffle=True); amp=dev.type=='cuda'; scl=torch.cuda.amp.GradScaler(enabled=amp)
            ba,bv,bt,pc=0,None,None,0
            for ep in range(params['epochs']):
                mdl.train()
                for bn,bc,by in ldr: opt.zero_grad(); 
                    with torch.cuda.amp.autocast(enabled=amp): ls=crit(mdl(bn,bc),by)
                    scl.scale(ls).backward();scl.unscale_(opt);torch.nn.utils.clip_grad_norm_(mdl.parameters(),1.0);scl.step(opt);scl.update()
                mdl.eval()
                with torch.no_grad():
                    with torch.cuda.amp.autocast(enabled=amp): vp=mdl(van,vac).cpu().numpy().flatten()
                    au=roc_auc_score(y.iloc[vi],vp);sched.step(au)
                    if au>ba: ba,bv,pc=au,vp.copy(),0; 
                        with torch.cuda.amp.autocast(enabled=amp): bt=mdl(ten,tec).cpu().numpy().flatten()
                    else: pc+=1
                    if pc>=params['patience']: break
            s_oof[vi]=bv;s_test+=bt/FOLDS
        print(f'NN {seed}: {roc_auc_score(y,s_oof):.6f}');oof+=s_oof/len(seeds);test+=s_test/len(seeds)
    print(f'NN Final: {roc_auc_score(y,oof):.6f}');return oof,test

print('Training functions ready')

In [ ]:
train = pd.read_csv('/kaggle/input/fiicode-2026-ai-competition/train.csv')
test = pd.read_csv('/kaggle/input/fiicode-2026-ai-competition/test.csv')
y = train[TARGET].astype(int)
print(f'Train: {len(train)}, Test: {len(test)}')

x_train, cats = build_features(train.drop(columns=[TARGET]))
x_test, _ = build_features(test)

print('\n=== CatBoost ===')
cb_oof, cb_test = train_cb(x_train, x_test, y, cats, CB_PARAMS, CB_SEEDS)

print('\n=== LightGBM ===')
lgbm_oof, lgbm_test = train_lgbm(x_train, x_test, y, cats, LGBM_PARAMS, LGBM_SEEDS)

print('\n=== AttentionNN ===')
nn_oof, nn_test = train_nn(x_train, x_test, y, cats, NN_PARAMS, NN_SEEDS)

# Triple blend: 60% CB + 15% LGBM + 25% NN
print('\n=== Triple Blend 60/15/25 ===')
blend_oof = 0.60*rank_norm(cb_oof) + 0.15*rank_norm(lgbm_oof) + 0.25*rank_norm(nn_oof)
blend_test = 0.60*rank_norm(cb_test) + 0.15*rank_norm(lgbm_test) + 0.25*rank_norm(nn_test)
print(f'Blend OOF AUC: {roc_auc_score(y, blend_oof):.6f}')

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: np.clip(blend_test, 1e-6, 1-1e-6)})
submission.to_csv('submission.csv', index=False)
print('Saved submission.csv')
submission.head()